This notebook aim to save and combine data akademik dan data pedoman akademik

### Data Pedoman Akademik (Dense Method)

In [1]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
        "https://akademik.nurulfikri.ac.id/2-administrasi/"
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
        "administrasi MBKM"
    ],
}

In [2]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [39]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_splits = text_splitter.split_documents(pages)


### Model Bahasa (IndoBert)

In [3]:
# memanggil Indobert dari transformer
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("Indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/Users/a/Programming/Langchain-Project/my-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# membuat class model embdding
from typing import List 
from langchain_core.embeddings import Embeddings
import torch

class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # polling token menjadi satu vector kalimat
        token_embeddings = outputs.last_hidden_state

        # melakukan mean polling
        sentence_embeddings = token_embeddings.mean(dim=1)

        # konversi ke list python
        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # metode untuk pencarian query pada chroma 
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)


In [5]:
embeddings = IndoBertEmbeddings()

In [6]:
from langchain_elasticsearch import ElasticsearchStore

In [7]:
from langchain_elasticsearch import DenseVectorStrategy


vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="langchain_index",
    embedding=embeddings,
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=DenseVectorStrategy(hybrid=True)
)

In [46]:
vector_store.add_documents(docs_splits)

2026-02-08 10:32:52,406 - INFO - HEAD http://localhost:9200/langchain_index [status:404 duration:0.004s]
2026-02-08 10:32:52,574 - INFO - PUT http://localhost:9200/langchain_index [status:200 duration:0.098s]
2026-02-08 10:32:58,166 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.256s]


['abdd8263-08a0-4601-8582-2c4021e50b0f',
 '4884d879-885e-4a6c-9205-d4b4c71ccaad',
 'e657b347-71ae-48ae-a4c7-d8655ee3a424',
 'e0765d45-70df-411e-8b22-9891ca453961',
 'd2c55c45-4364-41e2-bc80-82c26547ddbc',
 '7f317b18-bb94-4af7-86ed-45a5d7d6b707',
 'e7da3354-e82b-4f7b-9711-8a9af54ddd65',
 'e79575ed-6210-4056-93c8-5a9d46dcd6e3',
 '242edeec-861a-4396-a6a9-d517ccfe7606',
 'b4bbdefa-f46b-48bb-a221-e23342a0e761',
 'cf633ca5-1662-4d46-bdbb-fec2a5a3a0ee',
 '789803fb-5b75-4357-8802-f7a7ffde8e7e',
 '38de5c9d-8a6d-430d-98f5-e90375980552',
 'd8d5a929-0576-4930-8a4c-3f781f29d579',
 'd6edeb5d-0916-4300-a938-8922a06e3b1c',
 '0dc5a248-38dc-4c92-b709-357f40b387b3',
 '34e5b2d3-d2d2-4bf8-8c33-478311a7966f',
 '9abecbb7-b828-40bd-9213-73c9fb956e65',
 'a26522aa-3511-4ee5-81d1-bca0fa3f047a',
 '41568d81-46b7-413f-a170-f35696797331',
 '85237228-3c39-4412-b104-14015866bb4f',
 '2e185ece-0e4f-4dfe-bf5d-2c8edb5084d5',
 '21a2e366-4c0e-4e95-b447-2224cd93d765',
 'c5a642d9-867e-43bf-b498-c3b978177b97',
 'eb4d7d3e-832b-

### Testing Dense Retriever

In [8]:
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={
        "k":5
    }
)


In [9]:
retriever.invoke("Apa saja syarat untuk bisa lulus yah")

[Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/2-administrasi/', 'title': '2. Administrasi – Pedoman Akademik STT-NF', 'language': 'en-US'}, page_content='3. Update log catatan pada sheet yang disediakan di dalam folder tiap prodi\n4. Template Laporan Bisa di Download dilink berikut :\xa0Template Laporan Kegiatan MBKM\n*Format laporan boleh disesuaikan dengan format dari mitra\n5. Konfirmasi ke dosen pembimbing masing-masing segera setelah upload dokumen laporan selesai.\nInformasi Tentang Laporan MBKM\xa0\nPanduan Laporan MBKM dan Template Laporan MBKM\xa0:\xa0Panduan Laporan MBKM\nInformasi Tentang MBKM Magang diluar mitra dikti dan internal kampus :\nUntuk kelengkapan dokumen konversi silahkan arahkan mahasiswa untuk melengkapi dokumen berikut :\nbit.ly/suratMBKM\nAlur Pengajuan Surat :\n1. Buatlah Surat Pernyataan Mitra terlebih dahulu dan lengkapi dengan tanda tangan basah dan Cap Basah mitra. Kemudian Scan dokumen tersebut.\n2. Buat Surat Kesanggupan secara leng

### Data Akademik Mahasiswa (Sparse Method)

In [11]:
data_akademik = [
    '/Users/a/Programming/Langchain-Project/external-data/sintetik-data-akademik-mahasiswa (2).xlsx'
]

### Processing document

In [12]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

In [87]:
class DoclingLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [88]:
loader = DoclingLoader(data_akademik)

docs_akademik = loader.lazy_load()

In [89]:
data_akademik_split = text_splitter.split_documents(docs_akademik)

2026-02-05 12:58:33,875 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]
2026-02-05 12:58:33,956 - INFO - Going to convert document batch...
2026-02-05 12:58:33,957 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-02-05 12:58:33,958 - INFO - Processing document sintetik-data-akademik-mahasiswa (2).xlsx
2026-02-05 12:58:33,959 - INFO - Processing sheet 0: Data Mahasiswa Sintetik untuk R
2026-02-05 12:58:33,966 - INFO - Finished converting document sintetik-data-akademik-mahasiswa (2).xlsx in 0.10 sec.


In [10]:
vector_store_sparse = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="test_index",
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=ElasticsearchStore.BM25RetrievalStrategy(),
)

In [29]:
vector_store_sparse.add_documents(data_akademik_split)

2026-02-04 16:22:45,578 - INFO - HEAD http://localhost:9200/test_index [status:404 duration:0.003s]
2026-02-04 16:22:45,696 - INFO - PUT http://localhost:9200/test_index [status:200 duration:0.118s]
2026-02-04 16:22:45,727 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.030s]


['a90c9573-09f1-4a7a-9d88-e47600f46e3b',
 '12867448-e31f-4023-a83f-8cf6e5044d82',
 '4b5d04ac-6923-42fb-b640-9ce215e9e66f',
 '5d56f4dd-f242-4d5c-a7e3-8576dde01b7e',
 '265ef853-2ef3-4662-ad3f-026d612551f2',
 'b0e3227d-3077-48bf-b114-241972119a99',
 '7be5edd7-e1e7-4923-8c37-3262c54a2c8d',
 'f5b03288-c029-42aa-b00e-80dc41058277',
 'a29db36d-222d-493d-9260-45af2327037f',
 'f3cacb80-77e6-4c03-8bf2-6858595e1565',
 '7dbf0e00-4014-4e51-bfd0-766e48303b40',
 '22229420-9058-40e2-9d8b-af75326f5468',
 '2f7309be-c2b6-42db-b9ec-a2d9080cfd0b',
 '1e1a636c-74e8-44f1-b7c4-986a0395ac6b',
 'c5efe8db-e078-434f-90ff-69d600d46e4e',
 'a264992e-634d-48c8-8399-a2becd78aade',
 '560a19ab-914a-4a93-af84-c694126f1805']

In [13]:
vector_store_sparse.similarity_search("berapa ipk romi wahyudi")

[Document(metadata={}, page_content='|   17 | 2.021e+07   | Qori Handayani        | T. Informatika   |          5 |          20 |  3.15 |  3.2  |              87 | Aktif     |\n|   18 | 2.021e+07   | Romi Wahyudi Hasibuan | T. Informatika   |          5 |          18 |  2.5  |  2.8  |              75 | Aktif     |\n|   19 | 2.021e+07   | Siti Aminah           | T. Informatika   |          5 |          24 |  4    |  3.95 |              99 | Aktif     |\n|   20 | 2.021e+07   | Tono Suherman         | T. Informatika   |          5 |          15 |  0.5  |  1.8  |              30 | Non-Aktif |\n|   21 | 2.021e+07   | Usman Affandi         | T. Informatika   |          5 |          20 |  3.05 |  3.1  |              85 | Aktif     |\n|   22 | 2.021e+07   | Vina Panduwinata      | T. Informatika   |          5 |          22 |  3.55 |  3.6  |              93 | Aktif     |\n|   23 | 2.021e+07   | Wahyu Hidayat         | T. Informatika   |          5 |          20 |  2.85 |  3    |              8

### Create Agent Which can separate the work to do the conditioning

In [15]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import operator
from langchain_core.messages import AIMessage

In [16]:
class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], operator.add]

### Tools 

In [17]:

@tool
def query_from_academic_rule(query: str):
    """
    Gunakan tool ini HANYA untuk pertanyaan tentang ATURAN, KEBIJAKAN, SYARAT, atau PROSEDUR KAMPUS.
    JANGAN gunakan ini untuk mencari nama orang, nilai, atau data pribadi mahasiswa.
    Contoh input: "syarat yudisium", "aturan cuti", "biaya semester".
    """
    try:
        docs = vector_store.similarity_search(query, k=3)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data pedoman akademik."
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari Pedoman Akademik:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi Kesalahan saat mengakses vector database"


@tool
def get_student_academic_record(query: str):
    """
    Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.
    Termasuk: Nama lengkap, NIM (Nomor Induk Mahasiswa), IPK, Nilai, dan Status.
    Jika user bertanya "Siapa NIM dari Budi?", gunakan tool ini.
    Contoh input: "Budi Santoso", "Romi Wahyudi".
    """
    try:
        docs = vector_store_sparse.similarity_search(query)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data akademik mahasiswa"
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari data akademik mahasiswa:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi kesalahn saat mengakses data akademik"

In [28]:
import json
from langchain_core.messages import AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, END

class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model) # node 1 -> LLM thinking
        graph.add_node("action", self.take_action) # node 2 -> eksekusi tools
        
        # condional edge, True (llm -> action), false (LLM -> END)
        graph.add_conditional_edges( 
            "llm", self.exists_action, {True: "action", False: END} 
        )
        graph.add_edge("action", "llm") # edge action -> llm (balik lagi ke llm)
        graph.set_entry_point("llm") # node pintu masuk -> llm
        
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

   
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
    
        if not isinstance(result, AIMessage):
            return False

      
        if len(result.tool_calls) > 0:
            return True
            
        # cek apakah ada json dari hasil jawaban AI, if True need action or tools
        content = result.content if result.content else ""
        if '[{"name":' in content or "tool_calls" in content:
            print("🕵️ Terdeteksi JSON Tool Call di dalam teks!")
            return True
            
        return False

    def call_model(self, state: AgentState):
        messages = state["messages"]
        
        if len(messages) > 10:
             return {'messages': [AIMessage(content="Maaf, saya mencoba mencari tapi prosesnya terlalu lama (Loop detected). Mohon perjelas pertanyaan.")]}

        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        
        try:
            # Tambahkan print debug
            print("🤖 Model sedang berpikir...")
            response = self.model.invoke(messages)
            return {'messages': [response]}
        except Exception as e:
            print(f"Error invoke: {e}")
            return {'messages': [AIMessage(content="Error system.")]}

  
    def take_action(self, state: AgentState):
        last_message = state['messages'][-1]
        results = []
        tool_calls = []

        if hasattr(last_message, 'tool_calls') and len(last_message.tool_calls) > 0:
            tool_calls = last_message.tool_calls
            
        # manual parsing atau lihat tools yang dibutuhkan 
        else:
            try:
                content = last_message.content
                # Cari posisi kurung siku JSON [...]
                start_idx = content.find('[{"name":')
                if start_idx != -1:
                    json_str = content[start_idx:]
                    # Bersihkan jika ada sisa text di belakang (opsional)
                    end_idx = json_str.rfind('}]') + 2
                    json_str = json_str[:end_idx]
                    
                    parsed_tools = json.loads(json_str)
                    
                    # Konversi ke format standard tool call
                    for pt in parsed_tools:
                        tool_calls.append({
                            'name': pt['name'],
                            'args': pt['arguments'],
                            'id': 'manual_call' # ID dummy
                        })
            except Exception as e:
                print(f"❌ Gagal parsing manual JSON: {e}")

        # EKSEKUSI TOOLS
        for t in tool_calls:
            print(f"🛠️ Eksekusi Tool: {t['name']} dengan args: {t['args']}")
            
            if t['name'] not in self.tools:
                result = "Error: Tool name not found. Please check valid tools."
            else:
                try:
                    # Pastikan args adalah dict
                    args = t['args']
                    if isinstance(args, str):
                        args = json.loads(args)
                        
                    result = self.tools[t['name']].invoke(args)
                    
                    # Jika hasil kosong, beri tahu model secara eksplisit
                    if not result:
                        result = "Info: Data tidak ditemukan di database untuk input tersebut."
                        
                except Exception as e:
                    result = f"Error execution: {str(e)}"
            
            print(f"   📄 Hasil Tool: {str(result)[:100]}...") 

            results.append(ToolMessage(
                tool_call_id=t.get('id', 'manual_call'), 
                name=t['name'], 
                content=str(result)
            ))
            
        print("🔙 Kembali ke Model membawa data...")
        return {'messages': results}

### LLM model

In [29]:
from langchain_ollama import ChatOllama

In [30]:
system_prompt = """
Kamu adalah Asisten Akademik Kampus yang cerdas. Tugasmu adalah menjawab pertanyaan user.
Kamu memiliki akses ke dua alat:
1. `query_from_academic_rule`: Untuk mencari aturan umum (Pedoman).
2. `get_student_academic_record`: Untuk mencari data pribadi mahasiswa (Database).

STRATEGI ROUTING:
- Jika user bertanya ATURAN UMUM KAMPUS -> Gunakan `query_from_academic_rule`.
- Jika user bertanya DATA PRIBADI MAHASISWA -> Gunakan `get_student_academic_record`.
- Jika user bertanya KEDUANYA (misal: "Apakah saya memenuhi syarat?"), panggil KEDUA alat tersebut.
- Jika user hanya menyapa (Halo/Hi) -> JANGAN panggil alat, jawab langsung dengan sopan.
"""

In [31]:
model = ChatOllama(
    model="mistral:7b-instruct-v0.3-q8_0", 
    temperature=0, 
    streaming=True)

In [32]:
tools = [query_from_academic_rule, get_student_academic_record]

In [33]:
tools[1].name

'get_student_academic_record'

In [34]:
bot = Agent(model, tools, system=system_prompt)

In [ ]:
result = bot.graph.invoke({"messages": [HumanMessage(content="Apa tujuan Sekolah Tinggi Teknologi Terpadu Nurul Fikri melakukan kegiatan MBKM")]})

🤖 Model sedang berpikir...
🕵️ Terdeteksi JSON Tool Call di dalam teks!
🛠️ Eksekusi Tool: query_from_academic_rule dengan args: {'query': 'syarat lulus'}
   📄 Hasil Tool: Ditemukan informasi berikut dari Pedoman Akademik:
Add Article            
        







1. Syarat...
🔙 Kembali ke Model membawa data...
🤖 Model sedang berpikir...


In [36]:
print(result['messages'][-1].content)

 Untuk dapat lulus di Program Sarjana STT Terpadu Nurul Fikri, mahasiswa harus memenuhi persyaratan-persyaratan berikut:

1. Telah menyelesaikan mata kuliah minimal 148 SKS sesuai ketentuan program studi masing-masing.
2. Telah menyelesaikan mata kuliah Tugas Akhir.
3. Memiliki IPK minimal 2.00.
4. Memiliki sertifikat kompetensi.
5. Masa studi tidak melebihi 7 tahun / 14 semester.

Untuk melanjutkan studinya, jika masa studi telah melebihi 7 tahun (14 Semester), akan diberlakukan ketentuan SPP Progresif.


### Bagian ini digunakan untuk mengevaluasi sistem RAG, baik untuk proses `Retrieval`, maupun pengujian hasil `Generation` dari model LLM

In [24]:
import json
import numpy as np


def calculate_metric(retrieved_docs, ground_truth_source, k=5):
    top_k_docs = retrieved_docs[:k]

    # ambil sumber datanya (asumsi data pedoman akademik akan punya atribut sumber data)
    retrieved_sources = [doc.metadata.get('source') for doc in top_k_docs]

    # apakah URL yang benar ada di dalam list yang ditemukan
    if ground_truth_source in retrieved_sources:
        hit_score = 1
        recall_score = 1
    else:
        hit_score = 0
        recall_score = 0

    # berapa persen dokumen di Top K yang benar
    relevant_count = retrieved_sources.count(ground_truth_source)
    precision_score = relevant_count / k
    return hit_score, precision_score, recall_score

In [29]:
def evaluate_rag_system(dataset, retrieval_function, k_values=[1, 3, 5]):
    results = {k: {'hit_rate':[], 'precision':[], 'recall':[]} for k in k_values}

    for i, data in enumerate(dataset):
        query = data['question']
        gt_source = data['ground_truth_source']

        # query dengan vector store es
        retrieved_docs = retrieval_function.invoke(query)

        # hitung score
        for k in k_values:
            hit, prec, rec = calculate_metric(retrieved_docs, gt_source, k)

            results[k]['hit_rate'].append(hit)
            results[k]['precision'].append(prec)
            results[k]['recall'].append(rec)

    
    # rata-rata
    final_report = {}
    for k in k_values:
        final_report[f'Hit_Rate@{k}'] = np.mean(results[k]['hit_rate'])
        final_report[f'Precision{k}'] = np.mean(results[k]['precision'])
        final_report[f'Recall{k}'] = np.mean(results[k]['recall'])

    return final_report

In [43]:
dataset = [
    {
        "question": "Berapa minimal SKS untuk lulus di STT Terpadu Nurul Fikri",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/"
    },
    {
        "question": "Bisa jelaskan sejarah sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-sejarah/",
    },
    {
        "question": "Bisa berikan informasi mengenai Ahmad Rio Adriansyah",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/4-profil-dosen/"
    },
    {
        "question": "kapan evaluasi akademik dilaksanakan",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/"
    },
    {
        "question": "Apa etika mahasiswa terhadap dosen di sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/"
    }
]

In [44]:
report = evaluate_rag_system(dataset, retriever, [3, 5])

In [45]:
import pprint
pprint.pprint(report)

{'Hit_Rate@3': 0.8,
 'Hit_Rate@5': 0.8,
 'Precision3': 0.4666666666666666,
 'Precision5': 0.44000000000000006,
 'Recall3': 0.8,
 'Recall5': 0.8}
